<a href="https://colab.research.google.com/github/miiiray/banking-intent-ai/blob/main/notebooks/01_veri_inceleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

libraries and datas

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split  # scikit-learn kütüphanesinden veriyi bölmeye yarayan hazır fonksiyonu alıyoruz

url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"
# veriyi tablo olarak işlemek için
train = pd.read_csv(url + "train.csv")
test = pd.read_csv(url + "test.csv")
#verileri yükle

print(train.shape, test.shape)
train.head() #ilk 5ini görüntüle



(10003, 2) (3080, 2)


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


categories

In [ ]:
counts = train["category"].value_counts()

print(counts)
print("Kategori sayısı:", len(counts))

len(counts)
print(train.isnull().sum())
print("Tekrar eden satır:", train.duplicated().sum())


controls and cleaning

In [ ]:

# standartlaştırma
train_key = train["text"].fillna("").str.lower().str.split().str.join(" ")
test_key = test["text"].fillna("").str.lower().str.split().str.join(" ")

print("Train boş mesaj:", train_key.eq("").sum())
print("Test boş mesaj:", test_key.eq("").sum())

print("\nTest eksik değerler:")
print(test.isnull().sum())

print("\nTrain tekrar eden mesaj:", train_key.duplicated().sum())
print("Test tekrar eden mesaj:", test_key.duplicated().sum())

# Testte de bulunan eğitim satırlarını işaretliyoruz.
overlap = train_key.isin(test_key)

print("\nTestte de bulunan eğitim satırı:", overlap.sum())

temiz = train[~overlap & ~train_key.duplicated()]
print("Önce:", len(train), "Sonra:", len(temiz))


Diving the data

In [ ]:

# Temizlenmiş train verisini ikiye bölüyoruz:
# train_df -> modelin öğreneceği kısım (%85)
# val_df   -> ayarları seçmek için kullanacağımız deneme sınavı (%15)
train_df, val_df = train_test_split(
    temiz,                          # bölünecek veri: temizlenmiş train tablosu
    test_size=0.15,                 # verinin %15'i validation'a ayrılsın
    stratify=temiz["category"],     # her kategorinin oranı iki parçada da aynı kalsın
    random_state=42                 # bölme her çalıştırmada aynı çıksın (ikimiz de aynı sonucu alalım)
)

# Üç parçanın boyutlarını (satır, sütun) yazdırıp kontrol ediyoruz
print("Train:", train_df.shape)       # model bununla eğitilecek
print("Validation:", val_df.shape)    # ayarlar bununla seçilecek
print("Test:", test.shape)            # final sınavı, sadece en sonda kullanılacak



Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()                 # isimleri sayıya çeviren araç
encoder.fit(train_df["category"])        # 77 kategoriyi SADECE train'den öğren

y_train = encoder.transform(train_df["category"])   # isim -> sayı
y_val = encoder.transform(val_df["category"])
y_test = encoder.transform(test["category"])

print(train_df["category"].values[:3])   # ilk 3 kategori ismi
print(y_train[:3])                       # karşılık gelen sayılar
print("Sınıf sayısı:", len(encoder.classes_))


['top_up_failed' 'change_pin' 'getting_spare_card']
[59 22 40]
Sınıf sayısı: 77


Message Length Analysis

In [ ]:
uzunluk = train_df["text"].str.split().str.len()   # her mesajdaki kelime sayısı
print(uzunluk.describe())                          # ortalama, en kısa, en uzun
print("%95'lik değer:", uzunluk.quantile(0.95))    # mesajların %95'i bundan kısa


count    8493.000000
mean       11.982338
std         7.956291
min         2.000000
25%         7.000000
50%        10.000000
75%        13.000000
max        79.000000
Name: text, dtype: float64
%95'lik değer: 30.0


Tokenization and Padding

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

MAX_LEN = 30          # mesajların %95'i 30 kelimeden kısa
MAX_WORDS = 10000     # sözlükte en fazla 10.000 kelime

vectorizer = TextVectorization(
    max_tokens=MAX_WORDS,
    output_sequence_length=MAX_LEN   # kısa olanlara 0 ekle, uzunları kes
)

vectorizer.adapt(train_df["text"].values)   # sözlüğü SADECE train'den oluştur

X_train = vectorizer(train_df["text"].values)
X_val = vectorizer(val_df["text"].values)
X_test = vectorizer(test["text"].values)

print("Sözlükteki kelime sayısı:", len(vectorizer.get_vocabulary()))
print("X_train boyutu:", X_train.shape)
print(train_df["text"].values[0])
print(X_train[0].numpy())

Training Settings (shared by all models)

In [16]:
import time
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)   # rastgelelik sabit: her çalıştırmada aynı sonuç

VOCAB_SIZE = len(vectorizer.get_vocabulary())  # sözlükteki kelime sayısı
NUM_CLASSES = len(encoder.classes_)            # 77 kategori
EMBED_DIM = 128      # her kelimeyi 128 sayılık bir vektörle temsil et
EPOCHS = 30          # en fazla 30 tur eğit
BATCH_SIZE = 32      # model her seferde 32 mesaj görüp kendini güncellesin

print("Sözlük:", VOCAB_SIZE, "| Sınıf:", NUM_CLASSES)

Sözlük: 2301 | Sınıf: 77
